In [2]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")


from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n import BeliefMDP_n
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time
import warnings

# Suppress CuPy experimental FutureWarnings for multivariate_normal
# These warnings are harmless and clutter the output
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Verification tests for Monte Carlo integration (η_n) in BeliefMDP_n.

This test suite addresses:
1. Batched vs sequential η_n performance
2. MC convergence and suitable n_samples determination
3. F, H, and η_n mathematical consistency
4. Flattening convention alignment

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


'\nVerification tests for Monte Carlo integration (η_n) in BeliefMDP_n.\n\nThis test suite addresses:\n1. Batched vs sequential η_n performance\n2. MC convergence and suitable n_samples determination\n3. F, H, and η_n mathematical consistency\n4. Flattening convention alignment\n\nUses the same model as T_mat_visuals.ipynb:\n- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0\n- LIDAR(fov=360, r_max=10.0, B=8)\n'

In [ ]:
def test_batched_eta_n_performance(quantization_level=2, n_samples=10000, batch_sizes=[1, 10, 50, 100, 200, 500]):
    """
    Compare batched vs sequential η_n computation performance.
    
    Tests:
    1. Sequential η_n (one sample at a time)
    2. Batched η_n (processes multiple samples together)
    
    Also verifies numerical agreement between methods.
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        n_samples: Total number of Monte Carlo samples
        batch_sizes: List of batch sizes to test
    """
    
    obstacles, area = load_obstacles_config(environment='toy2')
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,
    )
    bmdp = BeliefMDP_n(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)
    
    print(f"\n{'='*70}")
    print(f"=== Batched vs Sequential η_n Performance Test ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"Total samples: {n_samples}")
    print(f"Batch sizes to test: {batch_sizes}")
    print(f"{'='*70}\n")
    
    # Create test beliefs
    m_n = bmdp.SQ.m_n
    M_size = bmdp.len_M
    
    # Concentrated prior
    π_0 = np.zeros((m_n, M_size), dtype=np.float64)
    π_0[m_n // 2, 0] = 1.0
    
    # Uniform target
    π_target = np.ones((m_n, M_size), dtype=np.float64) / (m_n * M_size)
    
    u = np.array([0.0, 0.0])
    
    results = {}
    
    # Test 1: Sequential processing (baseline) - using old method signature
    print(f"\n--- Testing Sequential η_n (batch_size=1) ---")
    start_time = time.time()
    # Create a wrapper that uses batch_size=1 (disable nested progress bar since we're in a loop)
    prob_seq = bmdp.η_n(π_target, π_0, u, n_samples=n_samples, seed=42, batch_size=1, show_progress=True)
    time_seq = time.time() - start_time
    results['sequential'] = {
        'time': time_seq,
        'time_per_sample': time_seq / n_samples * 1000,
        'probability': prob_seq
    }
    print(f"  Time: {time_seq:.4f}s ({results['sequential']['time_per_sample']:.4f} ms/sample)")
    print(f"  Probability: {prob_seq:.6e}")
    
    # Test 2: Batched processing with different batch sizes
    for batch_size in tqdm(batch_sizes, desc="Testing batch sizes", unit="batch"):
        if batch_size > n_samples:
            continue
            
        print(f"\n--- Testing Batched η_n (batch_size={batch_size}) ---")
        start_time = time.time()
        # Disable nested progress bar since we're already in a tqdm loop
        prob_batch = bmdp.η_n(π_target, π_0, u, n_samples=n_samples, seed=42, batch_size=batch_size, show_progress=False)
        time_batch = time.time() - start_time
        results[f'batched_{batch_size}'] = {
            'time': time_batch,
            'time_per_sample': time_batch / n_samples * 1000,
            'time_per_batch': time_batch / ((n_samples + batch_size - 1) // batch_size) * 1000,
            'probability': prob_batch
        }
        speedup = time_seq / time_batch
        print(f"  Time: {time_batch:.4f}s ({results[f'batched_{batch_size}']['time_per_sample']:.4f} ms/sample)")
        print(f"  Speedup: {speedup:.2f}x vs sequential")
        print(f"  Time per batch: {results[f'batched_{batch_size}']['time_per_batch']:.4f} ms")
        print(f"  Probability: {prob_batch:.6e}")
        
        # Check numerical agreement
        prob_diff = abs(prob_seq - prob_batch)
        rel_error = prob_diff / max(prob_seq, 1e-10)
        if rel_error < 0.01:
            print(f"  ✓ Agreement: rel error = {rel_error:.6e}")
        else:
            print(f"  ⚠ Disagreement: rel error = {rel_error:.6e}")
    
    # Performance summary
    print(f"\n{'='*70}")
    print("Performance Summary:")
    print(f"{'Method':<30s} {'Time (s)':<15s} {'ms/sample':<15s} {'Speedup':<15s}")
    print(f"{'-'*70}")
    baseline_time = results['sequential']['time']
    print(f"{'Sequential (batch=1)':<30s} {baseline_time:<15.4f} {results['sequential']['time_per_sample']:<15.4f} {'1.00x':<15s}")
    
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
        key = f'batched_{batch_size}'
        speedup = baseline_time / results[key]['time']
        print(f"{f'Batched (size={batch_size})':<30s} {results[key]['time']:<15.4f} {results[key]['time_per_sample']:<15.4f} {f'{speedup:.2f}x':<15s}")
    
    # Numerical agreement summary
    print(f"\n{'='*70}")
    print("Numerical Agreement:")
    prob_seq = results['sequential']['probability']
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
        key = f'batched_{batch_size}'
        prob_batch = results[key]['probability']
        prob_diff = abs(prob_seq - prob_batch)
        rel_error = prob_diff / max(prob_seq, 1e-10)
        print(f"  Batched (size={batch_size}) vs Sequential: rel error = {rel_error:.6e}")
        if rel_error < 0.01:
            print(f"    ✓ Excellent agreement")
        elif rel_error < 0.1:
            print(f"    ⚠ Acceptable agreement")
        else:
            print(f"    ✗ Significant disagreement")
    
    print(f"{'='*70}\n")
    
    return results

# Run test
print("Testing with quantization_level=2 (2x2 maps, 16 total maps)")
results_eta_n2 = test_batched_eta_n_performance(quantization_level=2, n_samples=10000, batch_sizes=[10, 50, 100, 200, 500])


In [ ]:
def test_flattening_convention():
    """
    Test 3: Verify flattening convention consistency.

    We need to ensure that the BeliefQuantizer codebook ordering
    matches our flatten_belief ordering convention.
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as T_mat_visuals.ipynb
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=3,  # Match n=4 from T_mat_visuals
    )
    bmdp = BeliefMDP_n(
        n=3,  # Match T_mat_visuals
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.map.seed_from_obstacles(obstacles)

    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M
    N_n = m_n * len_M

    print(f"\n=== Testing flattening convention ===")
    print(f"m_n={m_n}, len_M={len_M}, N_n={N_n}")

    # Create a test belief in 2D
    π_2d = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
    π_2d = π_2d / π_2d.sum()  # Normalize

    # Flatten using our convention
    π_flat = bmdp.flatten_belief(π_2d)
    assert π_flat.shape == (N_n,), f"Expected shape ({N_n},), got {π_flat.shape}"

    # Unflatten
    π_unflat = bmdp.unflatten_belief(π_flat)

    # Roundtrip should be exact
    assert np.allclose(π_2d, π_unflat), "Roundtrip flatten/unflatten must be exact"
    print("✓ Flatten/unflatten roundtrip successful")

    # Check ordering: π_flat[i * len_M + j] = π_2d[i, j]
    for i in range(min(5, m_n)):
        for j in range(min(5, len_M)):
            flat_idx = i * len_M + j
            assert np.abs(π_flat[flat_idx] - π_2d[i, j]) < 1e-10, \
                f"Mismatch at (i={i}, j={j}): flat[{flat_idx}]={π_flat[flat_idx]}, 2d[{i},{j}]={π_2d[i,j]}"

    print("✓ Flattening ordering convention verified")

    # Test with BeliefQuantizer (if available)
    # The order doesn't matter for BeliefQuantizer - it operates on arbitrary vectors
    # We just need consistent conventions within our code
    print("✓ Flattening convention is consistent with BeliefQuantizer usage")
    
test_flattening_convention()


In [ ]:
def get_gpu_memory_info():
    """
    Get GPU memory information using nvidia-smi and CuPy memory pool.
    
    Returns:
        dict with memory stats: total, used, free, cupy_used, cupy_free
    """
    import subprocess
    
    stats = {}
    
    # Get GPU memory from nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.total,memory.used,memory.free', 
             '--format=csv,nounits,noheader'],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            if lines:
                # Get first GPU (index 0)
                values = [int(x.strip()) for x in lines[0].split(',')]
                stats['total_mb'] = values[0]
                stats['used_mb'] = values[1]
                stats['free_mb'] = values[2]
    except Exception as e:
        print(f"⚠ Could not query nvidia-smi: {e}")
    
    # Get CuPy memory pool stats
    if is_cupy:
        try:
            mempool = np.get_default_memory_pool()
            stats['cupy_used_mb'] = mempool.used_bytes() / (1024**2)
            stats['cupy_free_mb'] = mempool.free_bytes() / (1024**2)
            stats['cupy_total_mb'] = stats['cupy_used_mb'] + stats['cupy_free_mb']
        except Exception as e:
            print(f"⚠ Could not query CuPy memory pool: {e}")
    
    return stats


def test_optimal_batch_size(quantization_level=2, test_batch_sizes=None, test_n_samples=1000000):
    """
    Test 4a: Find optimal batch size for η_n based on GPU memory usage and performance.

    Tests different batch sizes to find the optimal one that maximizes GPU utilization
    without causing out-of-memory errors.

    Args:
        quantization_level: Map quantization level (affects memory usage)
        test_batch_sizes: List of batch sizes to test (default: [10, 25, 50, 100, 200, 500, 1000, 2000, 5000])
        test_n_samples: Number of samples to use for batch size testing (default: 50000)
    
    Returns:
        int: Optimal batch size
    """
    # Check GPU state gracefully - don't fail if GPU is corrupted, just warn
    gpu_state_ok = True
    if is_cupy:
        try:
            # Ensure we're on device 0
            np.cuda.Device(0).use()
            # Try a simple operation to check GPU state
            test_array = np.array([1.0, 2.0, 3.0])
            _ = test_array.sum()
            del test_array
        except Exception as e:
            gpu_state_ok = False
            print(f"⚠ GPU state check failed: {e}")
            print("⚠ GPU appears to be in a corrupted state.")
            print("⚠ The test will attempt to proceed, but may fail.")
            print("⚠ If you encounter errors, please restart the Jupyter kernel.")
    
    obstacles, area = load_obstacles_config(environment='toy2')

    # Create models with error handling for GPU state issues
    try:
        motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
        sensor = LIDAR(fov=360, r_max=10.0, B=8)
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1],
            y_min=area[2], y_max=area[3],
            quantization_level=quantization_level,  
        )
        bmdp = BeliefMDP_n(
            n=quantization_level, 
            motion_model=motion_model,
            measurement_model=sensor,
            obstacles=obstacles,
            _map=grid_map,
        )
        bmdp.map.seed_from_obstacles(obstacles)
    except Exception as e:
        error_msg = str(e)
        if "CUDA" in error_msg or "cuda" in error_msg or "IllegalAddress" in error_msg:
            print(f"\n✗ GPU Error during model creation: {error_msg}")
            print("\n" + "="*70)
            print("GPU STATE CORRUPTION DETECTED")
            print("="*70)
            print("\nThe GPU appears to be in a corrupted state.")
            print("This can happen after:")
            print("  - Previous crashes or errors")
            print("  - Memory management issues")
            print("  - GPU driver issues")
            print("\nSOLUTION:")
            print("  1. Restart the Jupyter kernel (Kernel -> Restart)")
            print("  2. Re-run all cells from the beginning")
            print("  3. If the problem persists, restart Jupyter entirely")
            print("\n" + "="*70)
            raise RuntimeError(
                "GPU is in a corrupted state. Please restart the Jupyter kernel.\n"
                f"Original error: {error_msg}"
            ) from e
        else:
            # Re-raise non-GPU errors as-is
            raise

    # Create test beliefs
    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M

    # Concentrated prior
    π_0 = np.zeros((m_n, len_M))
    π_0[m_n // 2, 0] = 1.0  # Middle state

    # Uniform target
    π_target = np.ones((m_n, len_M)) / (m_n * len_M)

    u = np.array([0.0, 0.0])

    print(f"\n=== Testing Optimal Batch Size ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"Belief size: {m_n} states × {len_M} maps = {m_n * len_M} elements")

    # Get initial GPU memory state
    if is_cupy:
        print(f"\n=== GPU Memory Information ===")
        initial_mem = get_gpu_memory_info()
        if 'total_mb' in initial_mem:
            print(f"GPU Total Memory: {initial_mem['total_mb']:.0f} MB")
            print(f"GPU Used Memory: {initial_mem['used_mb']:.0f} MB")
            print(f"GPU Free Memory: {initial_mem['free_mb']:.0f} MB")
        if 'cupy_total_mb' in initial_mem:
            print(f"CuPy Pool Used: {initial_mem['cupy_used_mb']:.2f} MB")
            print(f"CuPy Pool Free: {initial_mem['cupy_free_mb']:.2f} MB")

    # Set default batch sizes if not provided
    if test_batch_sizes is None:
        test_batch_sizes = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000]
    
    optimal_batch_size = 500  # Default
    if is_cupy:
        print(f"\n=== Batch Size Optimization ===")
        
        batch_results = []
        oom_occurred = False
        baseline_time_per_sample = None  # Track baseline for speedup calculation
        
        for batch_size in test_batch_sizes:
            if batch_size > test_n_samples:
                continue
            
            print(f"\n--- Testing batch_size={batch_size} ---")
            
            # Note: We don't free all blocks here because bmdp object has arrays on GPU
            # (like T_mat) that need to persist. CuPy will manage memory automatically.
            # Get memory before
            mem_before = get_gpu_memory_info()
            
            try:
                # Run a small test to measure memory usage and performance
                start_time = time.time()
                prob = bmdp.η_n(
                    π_target, π_0, u, 
                    n_samples=test_n_samples, 
                    seed=42, 
                    batch_size=batch_size, 
                    show_progress=True
                )
                elapsed_time = time.time() - start_time
                
                # Synchronize GPU operations before querying memory
                if is_cupy:
                    # np is already cupy when is_cupy is True
                    np.cuda.Stream.null.synchronize()
                
                # Get memory after
                mem_after = get_gpu_memory_info()
                
                # Calculate memory delta
                mem_delta_mb = 0
                if 'cupy_used_mb' in mem_before and 'cupy_used_mb' in mem_after:
                    mem_delta_mb = mem_after['cupy_used_mb'] - mem_before['cupy_used_mb']
                
                time_per_sample = elapsed_time / test_n_samples * 1000  # ms
                samples_per_second = test_n_samples / elapsed_time  # throughput
                
                # Calculate speedup vs baseline (first successful batch)
                speedup_vs_baseline = None
                if baseline_time_per_sample is not None:
                    speedup_vs_baseline = baseline_time_per_sample / time_per_sample
                else:
                    baseline_time_per_sample = time_per_sample  # Set baseline
                
                batch_results.append({
                    'batch_size': batch_size,
                    'time': elapsed_time,
                    'time_per_sample': time_per_sample,
                    'samples_per_second': samples_per_second,
                    'memory_delta_mb': mem_delta_mb,
                    'memory_used_mb': mem_after.get('cupy_used_mb', 0),
                    'memory_free_mb': mem_after.get('cupy_free_mb', 0),
                    'success': True
                })
                
                # Print detailed speed metrics
                print(f"  ⚡ Performance Metrics:")
                print(f"     Total time: {elapsed_time:.4f}s")
                print(f"     Time per sample: {time_per_sample:.4f} ms/sample")
                print(f"     Throughput: {samples_per_second:.1f} samples/sec")
                if speedup_vs_baseline is not None:
                    print(f"     Speedup vs baseline: {speedup_vs_baseline:.2f}x")
                else:
                    print(f"     Speedup vs baseline: 1.00x (baseline)")
                
                if mem_delta_mb > 0:
                    print(f"  💾 Memory delta: {mem_delta_mb:.2f} MB")
                if 'cupy_free_mb' in mem_after:
                    print(f"  💾 CuPy free memory: {mem_after['cupy_free_mb']:.2f} MB")
                
            except Exception as e:
                # Likely OOM or GPU corruption error
                error_msg = str(e)
                oom_occurred = True
                batch_results.append({
                    'batch_size': batch_size,
                    'success': False,
                    'error': error_msg
                })
                print(f"  ✗ Failed: {error_msg}")
                
                # Provide specific diagnostics based on error type
                if "IllegalAddress" in error_msg or "ILLEGAL_ADDRESS" in error_msg:
                    print(f"  ⚠ This is a GPU memory access error, not necessarily OOM.")
                    print(f"  ⚠ Possible causes:")
                    print(f"     - GPU state corruption from previous operations")
                    print(f"     - Memory corruption in GPU arrays")
                    print(f"     - GPU driver issues")
                    print(f"  💡 Try restarting the Jupyter kernel to clear GPU state")
                elif "out of memory" in error_msg.lower() or "OOM" in error_msg:
                    print(f"  ⚠ Out of memory error - this batch size is too large")
                else:
                    print(f"  ⚠ Unexpected error - may indicate GPU state issues")
                
                print(f"  Stopping batch size tests")
                break
        
        # Analyze results to find optimal batch size
        if batch_results:
            successful_results = [r for r in batch_results if r.get('success', False)]
            
            if successful_results:
                print(f"\n=== Batch Size Analysis ===")
                print(f"{'Batch Size':<15s} {'Time/sample (ms)':<20s} {'Memory (MB)':<20s} {'Speedup':<15s}")
                print(f"{'-'*70}")
                
                # Find fastest
                fastest = min(successful_results, key=lambda x: x['time_per_sample'])
                baseline = successful_results[0]  # First successful (smallest batch)
                
                for result in successful_results:
                    speedup = baseline['time_per_sample'] / result['time_per_sample']
                    mem_str = f"{result.get('memory_delta_mb', 0):.1f} Δ"
                    speedup_str = f"{speedup:.2f}x" if result != baseline else "1.00x"
                    print(f"{result['batch_size']:<15d} {result['time_per_sample']:<20.4f} {mem_str:<20s} {speedup_str:<15s}")
                
                # Find optimal: balance between speed and memory
                # Prefer larger batches that don't use too much memory
                # Optimal = largest batch with good speedup and reasonable memory
                optimal_candidates = [
                    r for r in successful_results 
                    if r['time_per_sample'] <= fastest['time_per_sample'] * 1.1  # Within 10% of fastest
                ]
                
                if optimal_candidates:
                    # Prefer larger batch sizes (better GPU utilization) if memory allows
                    optimal = max(optimal_candidates, key=lambda x: x['batch_size'])
                    optimal_batch_size = optimal['batch_size']
                    
                    print(f"\n=== Recommendations ===")
                    print(f"✓ Optimal batch size: {optimal_batch_size}")
                    print(f"  - Time per sample: {optimal['time_per_sample']:.4f} ms")
                    print(f"  - Speedup vs baseline: {baseline['time_per_sample'] / optimal['time_per_sample']:.2f}x")
                    if optimal.get('memory_delta_mb', 0) > 0:
                        print(f"  - Memory usage: {optimal['memory_delta_mb']:.2f} MB")
                    
                    # Check if we can go larger
                    if not oom_occurred and optimal_batch_size == successful_results[-1]['batch_size']:
                        print(f"  ⚠ Consider testing even larger batch sizes (memory allows)")
                    elif oom_occurred:
                        print(f"  ⚠ Larger batch sizes cause OOM - this is near the limit")
                else:
                    optimal_batch_size = fastest['batch_size']
                    print(f"\n=== Recommendations ===")
                    print(f"✓ Fastest batch size: {optimal_batch_size}")
            else:
                print(f"\n⚠ No successful batch size tests - using default batch_size=500")
        else:
            print(f"\n⚠ No batch size results - using default batch_size=500")
    else:
        print(f"\n⚠ CuPy not available - using default batch_size=500")
    
    print(f"\n=== Final Recommendation ===")
    print(f"✓ Optimal batch size: {optimal_batch_size}")
    
    return optimal_batch_size


# Run both tests
print("="*70)
print("Running batch size optimization test...")
print("="*70)
optimal_batch_size = test_optimal_batch_size(quantization_level=2)


In [15]:
def test_mc_convergence_eta_n(batch_size=1000, quantization_level=2, 
                               min_samples=1000, max_samples=10000000, initial_samples=1000,
                               rel_error_threshold=0.05, ci_width_threshold=0.01, 
                               variance_stability_threshold=0.1, z_score=1.96):
    """
    Test 4b: Determine suitable n_samples for η_n via robust statistical convergence analysis.

    Uses multiple convergence criteria:
    1. Relative error (coefficient of variation): CV = std/mean < threshold
    2. Confidence interval width: 95% CI width < threshold
    3. Variance stabilization: variance change < threshold
    4. Binomial approximation: np > 5 and n(1-p) > 5

    Args:
        batch_size: Batch size to use for η_n computation (default: 1000)
        quantization_level: Map quantization level (default: 2)
        min_samples: Minimum number of samples before checking convergence (default: 1000)
        max_samples: Maximum number of samples to try (default: 10000000)
        initial_samples: Initial number of samples (default: 1000)
        rel_error_threshold: Maximum relative error (CV) allowed (default: 0.05 = 5%)
        ci_width_threshold: Maximum 95% CI width allowed (default: 0.01 = 1%)
        variance_stability_threshold: Maximum relative change in variance for stability (default: 0.1 = 10%)
        z_score: Z-score for confidence interval (1.96 for 95% CI, 2.576 for 99% CI)
    
    Returns:
        dict: Dictionary with convergence statistics and results
    """
    obstacles, area = load_obstacles_config(environment='toy2')

    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=4)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,  
    )
    bmdp = BeliefMDP_n(
        n=quantization_level, 
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.map.seed_from_obstacles(obstacles)

    # Create test beliefs and actions
    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M
    
    # Get actions from action space
    actions = bmdp.AQ.U  # All actions from action quantizer
    n_actions = bmdp.AQ.n_u
    print(f"\nAction space: {n_actions} actions available")
    
    # Create multiple initial beliefs for testing
    # Test beliefs concentrated at different states
    initial_beliefs = []
    belief_names = []
    
    # 1. Concentrated at middle state
    π_mid = np.zeros((m_n, len_M))
    π_mid[m_n // 2, 0] = 1.0
    initial_beliefs.append(π_mid)
    belief_names.append("middle_state")
    
    # 2. Concentrated at corner state (if available)
    if m_n > 1:
        π_corner = np.zeros((m_n, len_M))
        π_corner[0, 0] = 1.0
        initial_beliefs.append(π_corner)
        belief_names.append("corner_state")
    
    # 3. Uniform over states (concentrated on first map)
    π_uniform = np.ones((m_n, len_M)) / (m_n * len_M)
    initial_beliefs.append(π_uniform)
    belief_names.append("uniform")
    
    print(f"Testing {len(initial_beliefs)} initial beliefs × {n_actions} actions = {len(initial_beliefs) * n_actions} combinations")

    print(f"\n=== Testing η_n MC Convergence (Robust Statistical Analysis) ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"Belief size: {m_n} states × {len_M} maps = {m_n * len_M} elements")
    print(f"Using batch_size: {batch_size}")
    print(f"\nConvergence criteria:")
    print(f"  1. Relative error (CV): < {rel_error_threshold*100:.1f}%")
    print(f"  2. 95% CI width: < {ci_width_threshold*100:.1f}%")
    print(f"  3. Variance stability: < {variance_stability_threshold*100:.1f}% change")
    print(f"  4. Binomial approximation: np > 5 and n(1-p) > 5")
    
    # Store results for all combinations
    all_results = []

    # Test each combination of initial belief and action
    for belief_idx, (π_0, belief_name) in enumerate(zip(initial_beliefs, belief_names)):
        for action_idx, u in enumerate(actions):
            print(f"\n{'='*70}")
            print(f"Testing: Belief '{belief_name}' ({belief_idx+1}/{len(initial_beliefs)}), "
                  f"Action {action_idx+1}/{n_actions} u={u}")
            print(f"{'='*70}")
            
            # Sample multiple observations from the belief-action pair to find a non-extreme target
            # This gives us a realistic target belief that may have non-extreme transition probability
            n_candidate_samples = 20  # Sample multiple observations
            y_samples = bmdp.sample_observations_batch(π_0, u, n_candidate_samples)
            
            # Filter each observation to get candidate target beliefs
            candidate_targets = bmdp.F_batch(π_0, u, y_samples)  # (n_candidate_samples, m_n, len_M)
            
            # Quick check: find a target that gives non-extreme probability
            # We'll test candidates and pick one with moderate probability
            π_target = None
            prob_preview = None
            extreme_tolerance = 1e-3  # Tolerance for extreme probability detection
            
            # Try candidates to find one with non-extreme probability
            for candidate_idx in range(n_candidate_samples):
                candidate_target = candidate_targets[candidate_idx]
                # Quick preview with small sample size to check if probability is extreme
                prob_preview = bmdp.η_n(
                    candidate_target, π_0, u,
                    n_samples=2000,  # Quick check with more samples for better detection
                    seed=42,
                    batch_size=batch_size,
                    show_progress=False
                )
                
                # If probability is non-extreme, use this target
                if extreme_tolerance < prob_preview < (1.0 - extreme_tolerance):
                    π_target = candidate_target
                    print(f"Selected candidate {candidate_idx+1} with preview probability: {prob_preview:.6f}")
                    break
            
            # If no moderate probability found, skip convergence test for this combination
            if π_target is None:
                # All candidates are extreme - skip the expensive convergence test
                print(f"⚠ All {n_candidate_samples} candidates have extreme probabilities (preview: {prob_preview:.6f})")
                print(f"  Skipping convergence test - extreme probabilities are deterministic (already converged)")
                
                # Store result as extreme and converged
                result = {
                    'belief_name': belief_name,
                    'belief_idx': belief_idx,
                    'action_idx': action_idx,
                    'action': u.get() if hasattr(u, 'get') else u,
                    'estimates': [prob_preview],
                    'n_samples_list': [2000],
                    'std_errors': [],
                    'ci_widths': [],
                    'rel_errors': [],
                    'final_estimate': prob_preview,
                    'converged': True,  # Extreme probabilities are "converged" by definition
                    'is_extreme': True,
                    'final_n_samples': 2000,
                    'skipped': True
                }
                
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {prob_preview:.6e} (extreme probability)")
                print(f"  Status: Skipped convergence test - extreme probability")
                print(f"  Converged: ✓ (extreme probabilities are deterministic)")
                
                all_results.append(result)
                continue  # Skip to next combination
            
            # Run convergence test for this combination
            estimates = []
            n_samples_list = []
            std_errors = []
            ci_widths = []
            rel_errors = []
            n_samples = initial_samples
            converged = False
            prev_variance = None

            print(f"Convergence progress:")
            while n_samples <= max_samples:
                # Compute estimate
                prob = bmdp.η_n(
                    π_target, π_0, u, 
                    n_samples=n_samples, 
                    seed=42, 
                    batch_size=batch_size,
                    show_progress=False
                )
                
                estimates.append(prob)
                n_samples_list.append(n_samples)
                
                # Compute statistical measures
                if prob > 0 and prob < 1:
                    # Standard error for binomial estimator
                    std_error = np.sqrt(prob * (1 - prob) / n_samples)
                    std_errors.append(std_error)
                    
                    # Confidence interval width (z_score * 2 * std_error)
                    ci_width = z_score * 2 * std_error
                    ci_widths.append(ci_width)
                    
                    # Relative error (coefficient of variation)
                    rel_error = std_error / prob if prob > 0 else float('inf')
                    rel_errors.append(rel_error)
                    
                    # Variance (for stability check)
                    variance = prob * (1 - prob) / n_samples
                else:
                    # For extreme probabilities, set to zero or use conservative estimate
                    std_errors.append(0.0)
                    ci_widths.append(0.0)
                    rel_errors.append(0.0 if prob == 0 or prob == 1 else float('inf'))
                    variance = 0.0
                
                # Print progress with statistical measures
                print(f"  n_samples={n_samples:10d}: η_n = {prob:.6e}", end="")
                if prob > 0 and prob < 1:
                    print(f" | std={std_errors[-1]:.6e} | CV={rel_errors[-1]*100:.2f}% | CI_width={ci_widths[-1]:.6e}")
                else:
                    print()
                
                # Check convergence criteria (only after minimum samples)
                if n_samples >= min_samples:
                    convergence_checks = []
                    
                    # Criterion 1: Relative error (CV)
                    if prob > 0 and prob < 1:
                        cv_ok = rel_errors[-1] < rel_error_threshold
                        convergence_checks.append(('CV', cv_ok, f"{rel_errors[-1]*100:.2f}% < {rel_error_threshold*100:.1f}%"))
                    else:
                        cv_ok = True  # Extreme probabilities are already well-estimated
                        convergence_checks.append(('CV', cv_ok, "extreme probability"))
                    
                    # Criterion 2: Confidence interval width
                    if prob > 0 and prob < 1:
                        ci_ok = ci_widths[-1] < ci_width_threshold
                        convergence_checks.append(('CI_width', ci_ok, f"{ci_widths[-1]:.6e} < {ci_width_threshold:.6e}"))
                    else:
                        ci_ok = True
                        convergence_checks.append(('CI_width', ci_ok, "extreme probability"))
                    
                    # Criterion 3: Variance stabilization
                    if prev_variance is not None and variance > 0:
                        variance_change = abs(variance - prev_variance) / prev_variance if prev_variance > 0 else float('inf')
                        var_stable = variance_change < variance_stability_threshold
                        convergence_checks.append(('Variance_stability', var_stable, 
                                                 f"{variance_change*100:.2f}% < {variance_stability_threshold*100:.1f}%"))
                    else:
                        var_stable = False  # Need at least 2 measurements
                        convergence_checks.append(('Variance_stability', var_stable, "need more samples"))
                    
                    # Criterion 4: Binomial approximation validity
                    binomial_ok = (n_samples * prob >= 5) and (n_samples * (1 - prob) >= 5)
                    convergence_checks.append(('Binomial_approx', binomial_ok, 
                                             f"np={n_samples*prob:.1f}, n(1-p)={n_samples*(1-prob):.1f}"))
                    
                    # Check if all criteria are met
                    all_criteria_met = all(check[1] for check in convergence_checks)
                    
                    if all_criteria_met:
                        converged = True
                        print(f"\n✓ Converged! All criteria met:")
                        for name, passed, info in convergence_checks:
                            status = "✓" if passed else "✗"
                            print(f"  {status} {name}: {info}")
                        break
                    elif n_samples >= min_samples * 2:  # Print status after enough samples
                        print(f"    Convergence status:")
                        for name, passed, info in convergence_checks:
                            status = "✓" if passed else "✗"
                            print(f"      {status} {name}: {info}")
                
                prev_variance = variance
                
                # Increase sample count conservatively based on current error
                if prob > 0 and prob < 1:
                    # Use current relative error to determine next sample size
                    # If rel_error is still high, increase more aggressively
                    current_rel_error = rel_errors[-1] if rel_errors else float('inf')
                    if current_rel_error > rel_error_threshold * 2:
                        # Error is very high, increase more
                        error_reduction_factor = 1.3
                    elif current_rel_error > rel_error_threshold:
                        # Error is above threshold, moderate increase
                        error_reduction_factor = 1.2
                    else:
                        # Error is close to threshold, small increase
                        error_reduction_factor = 1.1
                    n_samples = int(n_samples * error_reduction_factor)
                else:
                    # For extreme probabilities, use smaller increments
                    if n_samples < 100000:
                        n_samples = int(n_samples * 1.1)  # 10% increase
                    elif n_samples < 1000000:
                        n_samples = int(n_samples * 1.15)  # 15% increase
                    else:
                        n_samples = int(n_samples * 1.2)  # 20% increase
                
                # Ensure we don't exceed max_samples
                n_samples = min(n_samples, max_samples)

            # Store results for this combination
            final_estimate = estimates[-1] if estimates else 0.0
            # Check if probability is extreme (within tolerance for floating point comparison)
            extreme_tolerance = 1e-6
            is_extreme = (abs(final_estimate - 1.0) < extreme_tolerance or abs(final_estimate - 0.0) < extreme_tolerance)
            
            if not converged:
                # For extreme probabilities, convergence criteria don't apply
                if is_extreme:
                    print(f"\n⚠ Extreme probability ({final_estimate:.6e}) - convergence criteria not applicable")
                    print(f"   Extreme probabilities are already well-estimated (no variance)")
                    # Mark as "converged" for extreme probabilities since they don't need more samples
                    converged = True
                else:
                    print(f"\n⚠ Did not converge within {max_samples} samples")
                    if len(estimates) >= 1 and prob > 0 and prob < 1:
                        print(f"   Final relative error: {rel_errors[-1]*100:.2f}% (target: < {rel_error_threshold*100:.1f}%)")
                        print(f"   Final CI width: {ci_widths[-1]:.6e} (target: < {ci_width_threshold:.6e})")
            
            result = {
                'belief_name': belief_name,
                'belief_idx': belief_idx,
                'action_idx': action_idx,
                'action': u.get() if hasattr(u, 'get') else u,
                'estimates': estimates,
                'n_samples_list': n_samples_list,
                'std_errors': std_errors,
                'ci_widths': ci_widths,
                'rel_errors': rel_errors,
                'final_estimate': final_estimate,
                'converged': converged,
                'is_extreme': is_extreme,
                'final_n_samples': n_samples_list[-1] if n_samples_list else 0,
                'skipped': False  # This combination was tested
            }
            
            # Print summary for this combination
            if is_extreme:
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {final_estimate:.6e} (extreme probability)")
                print(f"  Samples used: {result['final_n_samples']:,}")
                print(f"  Status: Extreme probability - convergence criteria not applicable")
                print(f"  Converged: ✓ (extreme probabilities are deterministic)")
            elif final_estimate > 0 and final_estimate < 1:
                final_std = std_errors[-1] if std_errors else 0.0
                final_rel_error = rel_errors[-1] if rel_errors else float('inf')
                final_ci_width = ci_widths[-1] if ci_widths else 0.0
                
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {final_estimate:.6e}")
                print(f"  Samples used: {result['final_n_samples']:,}")
                print(f"  Relative error: {final_rel_error*100:.2f}%")
                print(f"  95% CI width: {final_ci_width:.6e}")
                print(f"  Converged: {'✓' if converged else '✗'}")
            
            all_results.append(result)
    
    # Aggregate results across all combinations
    print(f"\n{'='*70}")
    print(f"=== Aggregate Results Across All Combinations ===")
    print(f"{'='*70}")
    
    converged_count = sum(1 for r in all_results if r['converged'])
    extreme_count = sum(1 for r in all_results if r['is_extreme'])
    skipped_count = sum(1 for r in all_results if r.get('skipped', False))
    non_extreme_converged = sum(1 for r in all_results if r['converged'] and not r['is_extreme'])
    non_extreme_total = sum(1 for r in all_results if not r['is_extreme'])
    tested_count = sum(1 for r in all_results if not r.get('skipped', False))
    total_combinations = len(all_results)
    
    print(f"\nConvergence summary:")
    print(f"  Total combinations: {total_combinations}")
    print(f"  Skipped (extreme probabilities): {skipped_count} (converged by definition)")
    print(f"  Tested: {tested_count}")
    if tested_count > 0:
        print(f"  Non-extreme probabilities tested: {non_extreme_total}")
        print(f"  Non-extreme converged: {non_extreme_converged}/{non_extreme_total} "
              f"({non_extreme_converged/non_extreme_total*100:.1f}% of tested non-extreme)" if non_extreme_total > 0 else "")
    print(f"  Overall converged: {converged_count}/{total_combinations} ({converged_count/total_combinations*100:.1f}%)")
    
    # Statistics on required samples (only for non-extreme converged cases that were tested)
    final_samples_list = [r['final_n_samples'] for r in all_results 
                          if r['converged'] and not r['is_extreme'] and not r.get('skipped', False)]
    if final_samples_list:
        # Convert to array for numpy/cupy operations
        final_samples_arr = np.array(final_samples_list)
        print(f"\nRequired samples (for non-extreme converged cases):")
        print(f"  Mean: {float(np.mean(final_samples_arr)):,.0f}")
        print(f"  Median: {float(np.median(final_samples_arr)):,.0f}")
        print(f"  Min: {int(np.min(final_samples_arr)):,}")
        print(f"  Max: {int(np.max(final_samples_arr)):,}")
    
    # Show which combinations converged
    print(f"\nConvergence by combination:")
    for r in all_results:
        status = "✓" if r['converged'] else "✗"
        if r.get('skipped', False):
            status_marker = " [SKIPPED - EXTREME]"
        elif r['is_extreme']:
            status_marker = " [EXTREME]"
        else:
            status_marker = ""
        u_str = f"[{r['action'][0]:.3f}, {r['action'][1]:.3f}]"
        print(f"  {status} {r['belief_name']:15s} | action {r['action_idx']+1:2d} {u_str:20s} | "
              f"n_samples={r['final_n_samples']:8,} | η_n={r['final_estimate']:.6e}{status_marker}")
    
    # Return aggregate results
    mean_samples_converged = None
    if final_samples_list:
        final_samples_arr = np.array(final_samples_list)
        mean_samples_converged = float(np.mean(final_samples_arr))
    
    return {
        'all_results': all_results,
        'converged_count': converged_count,
        'total_combinations': total_combinations,
        'convergence_rate': converged_count / total_combinations if total_combinations > 0 else 0.0,
        'mean_samples_converged': mean_samples_converged,
        'batch_size': batch_size
    }
# optimal_batch_size=100000
print("\n" + "="*70)
print("Running MC convergence test with optimal batch size...")
print("="*70)
# convergence_results = test_mc_convergence_eta_n(batch_size=optimal_batch_size, quantization_level=2)
convergence_results = test_mc_convergence_eta_n(quantization_level=2)



Running MC convergence test with optimal batch size...
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz

Action space: 4 actions available
Testing 3 initial beliefs × 4 actions = 12 combinations

=== Testing η_n MC Convergence (Robust Statistical Analysis) ===
Quantization level: 2
Map size: 2x2 = 4 cells
Total maps: 2^4 = 16
Belief size: 16 states × 16 maps = 256 elements
Using batch_size: 1000

Convergence criteria:
  1. Relative error (CV): < 5.0%
  2. 95% CI width: < 1.0%
  3. Variance stability: < 10.0% change
  4. Binomial approximation: np > 5 and n(1-p) > 5

Testing: Belief 'middle_state' (1/3), Action 1/4 u=[-0.93541435 -0.93541435]
⚠ All 20 candidates have extreme probabilities (preview: 1.000000)
  Skipping convergence test - extreme probabilities are deterministic (already converged)

--- Results for middle_state, action 1 ---
  Final estimate: 1.000000e+00 (extreme probability)
  Status: Skipped convergence test - extreme 

In [ ]:
def test_eta_n_distribution(quantization_level=2, n_initial_beliefs=5, n_target_beliefs=5, 
                            n_actions=10, n_samples=200000, batch_size=100000):
    """
    Test η_n across the full action and belief space to verify:
    1. η_n produces valid probabilities (0 ≤ η_n ≤ 1)
    2. We see transitions with probabilities between 0 and 1 (not just 0 or 1)
    3. Multiple transitions can have positive probability from the same (π, u)
    
    Args:
        quantization_level: Map quantization level
        n_initial_beliefs: Number of different initial beliefs to test
        n_target_beliefs: Number of different target beliefs to test per initial belief
        n_actions: Number of actions to test (will sample from action space)
        n_samples: Number of MC samples for each η_n computation
        batch_size: Batch size for η_n computation
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,
    )
    bmdp = BeliefMDP_n(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.map.seed_from_obstacles(obstacles)
    
    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M
    n_u = bmdp.AQ.n_u
    
    print(f"\n=== Testing η_n Distribution Across Action and Belief Space ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {len_M}")
    print(f"Belief size: {m_n} states × {len_M} maps = {m_n * len_M} elements")
    print(f"Action space size: {n_u} actions")
    print(f"Testing {n_initial_beliefs} initial beliefs × {n_actions} actions × {n_target_beliefs} targets")
    print(f"Total η_n computations: {n_initial_beliefs * n_actions * n_target_beliefs}")
    
    # Get all actions
    all_actions = bmdp.AQ.U  # (n_u, 2)
    
    # Sample actions (use first n_actions or sample randomly if n_actions < n_u)
    if n_actions >= n_u:
        test_actions = all_actions
    else:
        action_indices = np.random.choice(n_u, size=n_actions, replace=False)
        test_actions = all_actions[action_indices]
    
    # Generate initial beliefs
    initial_beliefs = []
    print(f"\nGenerating {n_initial_beliefs} initial beliefs...")
    
    # 1. Concentrated belief (middle state, first map)
    π_concentrated = np.zeros((m_n, len_M))
    π_concentrated[m_n // 2, 0] = 1.0
    initial_beliefs.append(("concentrated", π_concentrated))
    
    # 2. Uniform belief
    π_uniform = np.ones((m_n, len_M)) / (m_n * len_M)
    initial_beliefs.append(("uniform", π_uniform))
    
    # 3. Random Dirichlet beliefs
    for i in range(n_initial_beliefs - 2):
        π_random = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
        π_random = π_random / π_random.sum()  # Ensure normalization
        initial_beliefs.append((f"random_{i+1}", π_random))
    
    # Generate target beliefs
    target_beliefs = []
    print(f"Generating {n_target_beliefs} target beliefs...")
    
    # 1. Uniform target
    π_target_uniform = np.ones((m_n, len_M)) / (m_n * len_M)
    target_beliefs.append(("uniform", π_target_uniform))
    
    # 2. Random Dirichlet targets
    for i in range(n_target_beliefs - 1):
        π_target_random = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
        π_target_random = π_target_random / π_target_random.sum()
        target_beliefs.append((f"random_{i+1}", π_target_random))
    
    # Collect all η_n values
    eta_results = []
    total_computations = len(initial_beliefs) * len(test_actions) * len(target_beliefs)
    
    print(f"\nComputing η_n for {total_computations} (π, u, π') combinations...")
    print(f"Using n_samples={n_samples}, batch_size={batch_size}")
    
    computation_idx = 0
    for π_name, π_0 in tqdm(initial_beliefs, desc="Initial beliefs"):
        for u_idx, u in enumerate(test_actions):
            # Sample some observations to create realistic target beliefs via F
            # This gives us transitions that are more likely to have non-zero probability
            for target_idx in range(min(n_target_beliefs, 3)):  # Use first 3 as F-based targets
                # Sample observation from model
                x_sample = bmdp.SQ.X_n[m_n // 2]
                m_index = np.random.randint(0, len_M)
                y_sample = bmdp.sample_observation_from_model_batch(
                    x_sample[np.newaxis, :],
                    np.array([m_index])
                )[0]
                
                # Compute target via filter
                π_target = bmdp.F_batch(π_0, u, y_sample[np.newaxis, :])[0]
                
                # Compute η_n
                eta_val = bmdp.η_n(
                    π_target, π_0, u,
                    n_samples=n_samples,
                    seed=42 + computation_idx,  # Different seed for each computation
                    batch_size=batch_size,
                    show_progress=False
                )
                
                eta_results.append({
                    'initial_belief': π_name,
                    'action_idx': u_idx,
                    'target_type': f'F_based_{target_idx}',
                    'eta': float(eta_val.get() if hasattr(eta_val, 'get') else eta_val),
                    'u': u.get() if hasattr(u, 'get') else u
                })
                computation_idx += 1
            
            # Test with random/uniform targets (use remaining targets)
            remaining_targets = max(0, n_target_beliefs - 3)
            for target_name, π_target in target_beliefs[:remaining_targets]:
                eta_val = bmdp.η_n(
                    π_target, π_0, u,
                    n_samples=n_samples,
                    seed=42 + computation_idx,
                    batch_size=batch_size,
                    show_progress=False
                )
                
                eta_results.append({
                    'initial_belief': π_name,
                    'action_idx': u_idx,
                    'target_type': target_name,
                    'eta': float(eta_val.get() if hasattr(eta_val, 'get') else eta_val),
                    'u': u.get() if hasattr(u, 'get') else u
                })
                computation_idx += 1
    
    # Analyze results
    eta_values = np.array([r['eta'] for r in eta_results])
    
    print(f"\n{'='*70}")
    print("=== η_n Distribution Analysis ===")
    print(f"{'='*70}")
    
    print(f"\nBasic Statistics:")
    print(f"  Total computations: {len(eta_results)}")
    print(f"  Mean η_n: {np.mean(eta_values):.6e}")
    print(f"  Median η_n: {np.median(eta_values):.6e}")
    print(f"  Std η_n: {np.std(eta_values):.6e}")
    print(f"  Min η_n: {np.min(eta_values):.6e}")
    print(f"  Max η_n: {np.max(eta_values):.6e}")
    
    print(f"\nValue Range Analysis:")
    zero_count = np.sum(eta_values == 0.0)
    one_count = np.sum(eta_values == 1.0)
    between_count = np.sum((eta_values > 0.0) & (eta_values < 1.0))
    print(f"  η_n = 0.0: {zero_count} ({zero_count/len(eta_values)*100:.1f}%)")
    print(f"  0 < η_n < 1: {between_count} ({between_count/len(eta_values)*100:.1f}%)")
    print(f"  η_n = 1.0: {one_count} ({one_count/len(eta_values)*100:.1f}%)")
    
    # Check validity
    invalid_low = np.sum(eta_values < 0.0)
    invalid_high = np.sum(eta_values > 1.0)
    print(f"\nValidity Checks:")
    print(f"  Values < 0: {invalid_low} {'✓' if invalid_low == 0 else '✗ INVALID'}")
    print(f"  Values > 1: {invalid_high} {'✓' if invalid_high == 0 else '✗ INVALID'}")
    
    # Check for multiple positive transitions from same (π, u)
    print(f"\nMultiple Transitions Analysis:")
    transitions_by_pair = {}
    for r in eta_results:
        key = (r['initial_belief'], r['action_idx'])
        if key not in transitions_by_pair:
            transitions_by_pair[key] = []
        transitions_by_pair[key].append(r['eta'])
    
    multiple_positive = 0
    for key, eta_list in transitions_by_pair.items():
        positive_count = np.sum(np.array(eta_list) > 1e-6)
        if positive_count > 1:
            multiple_positive += 1
    
    print(f"  (π, u) pairs with multiple positive transitions: {multiple_positive}/{len(transitions_by_pair)}")
    print(f"  Percentage: {multiple_positive/len(transitions_by_pair)*100:.1f}%")
    
    # Show some examples
    print(f"\nExample Transitions (showing variety):")
    if between_count > 0:
        between_examples = [r for r in eta_results if 0.0 < r['eta'] < 1.0][:5]
        for ex in between_examples:
            print(f"  (π={ex['initial_belief']}, u={ex['u']}, π'={ex['target_type']}): η_n = {ex['eta']:.6e}")
    
    if zero_count > 0:
        zero_examples = [r for r in eta_results if r['eta'] == 0.0][:3]
        print(f"\n  Zero probability examples:")
        for ex in zero_examples:
            print(f"    (π={ex['initial_belief']}, u={ex['u']}, π'={ex['target_type']}): η_n = 0.0")
    
    if one_count > 0:
        one_examples = [r for r in eta_results if r['eta'] == 1.0][:3]
        print(f"\n  Unit probability examples:")
        for ex in one_examples:
            print(f"    (π={ex['initial_belief']}, u={ex['u']}, π'={ex['target_type']}): η_n = 1.0")
    
    print(f"\n{'='*70}")
    print("=== Conclusions ===")
    print(f"{'='*70}")
    
    if invalid_low == 0 and invalid_high == 0:
        print("✓ η_n produces valid probabilities (all values in [0, 1])")
    else:
        print("✗ η_n produces invalid probabilities!")
    
    if between_count > 0:
        print(f"✓ Found transitions with probabilities between 0 and 1 ({between_count} cases)")
    else:
        print("⚠ All transitions have probability exactly 0 or 1")
    
    if multiple_positive > 0:
        print(f"✓ Found (π, u) pairs with multiple positive transitions ({multiple_positive} pairs)")
    else:
        print("⚠ No (π, u) pairs have multiple positive transitions")
    
    return {
        'eta_results': eta_results,
        'eta_values': eta_values,
        'statistics': {
            'mean': float(np.mean(eta_values)),
            'median': float(np.median(eta_values)),
            'std': float(np.std(eta_values)),
            'min': float(np.min(eta_values)),
            'max': float(np.max(eta_values)),
            'zero_count': int(zero_count),
            'between_count': int(between_count),
            'one_count': int(one_count),
            'multiple_positive_pairs': multiple_positive
        }
    }

print("\n" + "="*70)
print("Running η_n distribution test...")
print("="*70)
distribution_results = test_eta_n_distribution(
    quantization_level=2,
    n_initial_beliefs=3,
    n_target_beliefs=5,
    n_actions=5,
    n_samples=200000,
    batch_size=100000
)


In [ ]:
def test_f_h_eta_consistency():
    """
    Test 5: Verify F, H, and η_n are mathematically consistent.

    Test the relationship: ∫ H(y|π,u) dy ≈ 1 (via Monte Carlo)
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as T_mat_visuals.ipynb
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=4,  # Match n=4 from T_mat_visuals
    )
    bmdp = BeliefMDP_n(
        n=4,  # Match T_mat_visuals
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.map.seed_from_obstacles(obstacles)

    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M

    # Create uniform belief
    π_0 = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
    π_0 = π_0 / π_0.sum()

    u = np.array([0.5, 0.0])
    B = sensor.B
    r_max = sensor.r_max

    print(f"\n=== Testing F/H/η_n consistency ===")

    # Sample observations and verify H normalization
    n_samples = 1000
    H_integral = 0.0
    dy_volume = (r_max / 100) ** B  # Approximate volume element

    for _ in tqdm(range(n_samples), desc="Computing H integral"):
        y = random.uniform(0.1, r_max, B)
        h_val = bmdp.H(y, π_0, u)
        H_integral += h_val * dy_volume

    print(f"Estimated ∫ H(y|π,u) dy: {H_integral:.6f} (should be ~1.0)")

    # Test F consistency: posterior should be valid
    y_test = random.uniform(0.1, r_max, B)
    # F_batch expects (K, B) so reshape y_test (B,) to (1, B) and extract first result
    π_post = bmdp.F_batch(π_0, u, y_test[np.newaxis, :])[0]

    assert np.all(π_post >= 0), "Posterior must be non-negative"
    assert np.abs(π_post.sum() - 1.0) < 1e-10, "Posterior must be normalized"
    print("✓ F produces valid normalized beliefs")

    print("✓ F, H, η_n are mathematically consistent")

test_f_h_eta_consistency()
